In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_validate, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, make_scorer, matthews_corrcoef
import matplotlib.pyplot as plt

In [3]:
from tqdm import tqdm

# Charger les deux fichiers
full_features = pd.read_csv("final_sequence_dataset_features.csv")
df_labels = pd.read_csv("Dataset_gbm.csv")

# Certaines mutations apparaissent plusieurs fois dans les datasets : on enlève les doublons 
df_features_clean = full_features.drop_duplicates(subset=['UniProt ID', 'Mutation'])
df_labels_clean = df_labels.drop_duplicates(subset=['UniProt ID', 'Mutation'])
print(f"Duplicates in the features : {full_features.duplicated(subset=['UniProt ID', 'Mutation']).sum()}")
print(f"Duplicates in the labels : {df_labels.duplicated(subset=['UniProt ID', 'Mutation']).sum()}")

#  Fusionner les DataFrames (Inner Join)
# Cela ne garde que les mutations présentes dans les deux fichiers
df_final = pd.merge(df_features_clean, 
                    df_labels_clean[['UniProt ID', 'Mutation', 'Class']], 
                    on=['UniProt ID', 'Mutation'], 
                    how='inner')

# Vérification
print(f"Features originales : {len(full_features)}")
print(f"Après fusion avec labels : {len(df_final)}")
print(f"Répartition des classes :\n{df_final['Class'].value_counts()}")


# Separate the full dataset into 3 subdatasets 
# 0 = Helix (Alpha), 1 = Strand (Beta), 2 = Coil
df_alpha = df_final[df_final['secondary_structure'] == 0].copy()
df_beta  = df_final[df_final['secondary_structure'] == 1].copy()
df_coil  = df_final[df_final['secondary_structure'] == 2].copy()

# Show lengths of each subdataset 
print(f"Total mutations : {len(df_final)}")
print("-" * 30)
print(f"Beta-Strand (1) : {len(df_beta)} mutations")

y = df_beta['Class'].map({'Driver': 1, 'Passenger': 0})
# 2. Identifier les colonnes à supprimer absolument
# On retire l'ID, la Mutation, les séquences brutes (tri, n_M, etc.) 
# et les odds ratios flottants (nM_odds_rat, etc.)
cols_to_drop = [
    'UniProt ID', 'Gene Name', 'Mutation', 'Wild', 'Mut', 'Pos', 'Class', 
    'secondary_structure', 'tri', 'n_M', 'M_c', 'n__M', 'M__c', 'nM', 'Mc'
]

# On ajoute aussi tous les odds ratios non-codés (flottants) à la liste de suppression
or_float_cols = [c for c in df_beta.columns if 'odds_rat' in c or 'odds_ra' in c or 'odds_ratio' in c]
cols_to_drop.extend(or_float_cols)

# 3. Création de X final
X = df_beta.drop(columns=[c for c in cols_to_drop if c in df_beta.columns])

# Split Train/Test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardisation
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=X.columns)

# 2. Sélection des caractéristiques (Méthode 'feat_importances.nlargest')
#on utilise ExtraTreesClassifier pour calculer l'importance des features, car on ne peut pas récuperer les features les plus importantes à partir d'un SVM
print("--- Phase de calcul de l'importance des features ---")
forest = ExtraTreesClassifier(n_estimators=250, random_state=42, n_jobs=-1)
forest.fit(X_train, y_train)
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)

# 3. Comparaison des hyperparamètres
configs = {
    "User_Optimized (C=1)": {"C": 1, "kernel": "rbf", "gamma": "scale"},
    "Article_Reference (C=10)": {"C": 10, "kernel": "rbf", "gamma": 0.1}
}

# Nous testons l'impact du nombre de features (N)
results = []
for n_features in tqdm([i for i in range(1, 31)], desc="Testing feature counts"):
    selected_cols = importances.head(n_features).index
    X_train_sel = X_train[selected_cols]
    
    for name, params in configs.items():
        model = SVC(probability=True, **params)
        # scoring='roc_auc' permet de récupérer directement l'AUC
        cv_auc_scores = cross_val_score(model, X_train_sel, y_train, cv=10, scoring='roc_auc', n_jobs=-1)
        
        # On peut aussi récupérer l'accuracy si besoin
        cv_acc_scores = cross_val_score(model, X_train_sel, y_train, cv=10, scoring='accuracy', n_jobs=-1)

        results.append({
            "Config": name,
            "N_Features": n_features,
            "Accuracy": cv_acc_scores.mean(), # Moyenne des 10 folds
            "AUC": cv_auc_scores.mean(),     # Moyenne des 10 folds
            "CV_AUC_Std": cv_auc_scores.std()    # Écart-type pour vérifier la stabilité
        })

df_results = pd.DataFrame(results)
print("\n--- Rapport de performance pour les Beta-Strands ---")
print(df_results.pivot(index="N_Features", columns="Config", values=["Accuracy", "AUC"]))

Duplicates in the features : 21
Duplicates in the labels : 23
Features originales : 16928
Après fusion avec labels : 16907
Répartition des classes :
Class
Passenger    8703
Driver       8204
Name: count, dtype: int64
Total mutations : 16907
------------------------------
Beta-Strand (1) : 1931 mutations
--- Phase de calcul de l'importance des features ---


Testing feature counts: 100%|██████████| 30/30 [03:16<00:00,  6.54s/it]


--- Rapport de performance pour les Beta-Strands ---
                           Accuracy                       \
Config     Article_Reference (C=10) User_Optimized (C=1)   
N_Features                                                 
1                          0.600411             0.598463   
2                          0.637357             0.634755   
3                          0.636054             0.647080   
4                          0.640649             0.640637   
5                          0.635446             0.635463   
6                          0.626385             0.638027   
7                          0.647105             0.657415   
8                          0.638018             0.648349   
9                          0.626347             0.647708   
10                         0.616644             0.647700   
11                         0.631533             0.649623   
12                         0.623762             0.652245   
13                         0.627654           

In [4]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def balanced_accuracy(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return (sensitivity + specificity) / 2

# --- ÉTAPE 4 : FINALISATION DU MODÈLE BETA (N=10, C=1) ---

# 1. Récupération des 10 meilleures caractéristiques identifiées par ExtraTrees
#On a choisit de garder les 10 meilleurs sur base du meilleur AUC obtenu précédement
top_11_features = importances.head(11).index.tolist()
print("\nTop 11 features sélectionnées pour le modèle SVM Beta-Strand :")
for i, feat in enumerate(top_11_features, 1):
    print(f"{i:2d}. {feat}")

# 2. Filtrage des données pour ne garder que ces features
X_train_final = X_train[top_11_features]
X_test_final = X_test[top_11_features]

# 3. Entraînement du modèle SVM final avec tes hyperparamètres optimisés
print("\n Entraînement du modèle en cours...")
final_svm_beta = SVC(
    C=1, 
    kernel='rbf', 
    gamma='scale', 
    probability=True, 
    random_state=42
)
final_svm_beta.fit(X_train_final, y_train)

# 4. Évaluation finale pour confirmation
y_pred_train = final_svm_beta.predict(X_train_final)
y_proba_train = final_svm_beta.predict_proba(X_train_final)[:, 1]


scoring = {'accuracy': 'accuracy','sensitivity': 'recall','specificity': make_scorer(specificity_score),'mcc': make_scorer(matthews_corrcoef),'auc': 'roc_auc', 'balanced_accuracy': make_scorer(balanced_accuracy)}
cv = KFold(n_splits=10, shuffle=True, random_state=42)
cv_results = cross_validate(final_svm_beta, X_train_final, y_train, cv=cv, scoring=scoring, n_jobs=-1)

y_pred_test = final_svm_beta.predict(X_test_final)
y_proba_test = final_svm_beta.predict_proba(X_test_final)[:, 1]

print("\n--- Performance Metrics For our SVM Beta-Strand Model ---")
print()
print(f"Number of mutations in the train set : {len(y_train)} with {y_train.sum()} drivers and {len(y_train) - y_train.sum()} passengers")
print(f"Number of mutations in the test set : {len(y_test)} with {y_test.sum()} drivers and {len(y_test) - y_test.sum()} passengers")
print()
#metrics on the train set-----------------------------------------------------
train_auc = roc_auc_score(y_train, y_proba_train)
true_positives = confusion_matrix(y_train, y_pred_train)[1, 1]
true_negatives = confusion_matrix(y_train, y_pred_train)[0, 0]
false_positives = confusion_matrix(y_train, y_pred_train)[0, 1]
false_negatives = confusion_matrix(y_train, y_pred_train)[1, 0]
train_accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
train_sensitivity = true_positives / (true_positives + false_negatives)
train_specificity = true_negatives / (true_negatives + false_positives)
train_balanced_accuracy = (train_sensitivity + train_specificity) / 2
train_MCC = (true_positives * true_negatives - false_positives * false_negatives) / np.sqrt((true_positives + false_positives) * (true_positives + false_negatives) * (true_negatives + false_positives) * (true_negatives + false_negatives))

# metrics on the test set-----------------------------------------------------
true_positives = confusion_matrix(y_test, y_pred_test)[1, 1]
true_negatives = confusion_matrix(y_test, y_pred_test)[0, 0]
false_positives = confusion_matrix(y_test, y_pred_test)[0, 1]
false_negatives = confusion_matrix(y_test, y_pred_test)[1, 0]
test_auc = roc_auc_score(y_test, y_proba_test)
test_accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
test_sensitivity = true_positives / (true_positives + false_negatives)
test_specificity = true_negatives / (true_negatives + false_positives)
test_balanced_accuracy = (test_sensitivity + test_specificity) / 2
test_MCC = (true_positives * true_negatives - false_positives * false_negatives) / np.sqrt((true_positives + false_positives) * (true_positives + false_negatives) * (true_negatives + false_positives) * (true_negatives + false_negatives))

print(f"Sensitivity        : train set = {train_sensitivity:.4f}  |  CV = {np.mean(cv_results['test_sensitivity']):.4f} | test set = {test_sensitivity:.4f}")
print(f"Specificity        : train set = {train_specificity:.4f}  |  CV = {np.mean(cv_results['test_specificity']):.4f} | test set = {test_specificity:.4f}")
print(f"Accuracy           : train set = {train_accuracy:.4f}  |  CV = {np.mean(cv_results['test_accuracy']):.4f} | test set = {test_accuracy:.4f}")
print(f"MCC                : train set = {train_MCC:.4f}  |  CV = {np.mean(cv_results['test_mcc']):.4f} | test set = {test_MCC:.4f}")
print(f"AUC                : train set = {train_auc:.4f}  |  CV = {np.mean(cv_results['test_auc']):.4f} | test set = {test_auc:.4f}")
print(f"balanced accuracy  : train set = {train_balanced_accuracy:.4f}  |  CV = {np.mean(cv_results['test_balanced_accuracy']):.4f} | test set = {test_balanced_accuracy:.4f}")


Top 11 features sélectionnées pour le modèle SVM Beta-Strand :
 1.  rsa
 2. risj880101
 3. dG_numeric_diff
 4. JUKT750101_normalize_diff
 5. MAXF760106_normalize
 6. niek910102
 7. Br_normalize_diff
 8. K0_normalize
 9. dHc_numeric_diff
10. dHc_normalize_diff
11. -TdSc_normalize_diff

 Entraînement du modèle en cours...

--- Performance Metrics For our SVM Beta-Strand Model ---

Number of mutations in the train set : 1544 with 854 drivers and 690 passengers
Number of mutations in the test set : 387 with 214 drivers and 173 passengers

Sensitivity        : train set = 0.7611  |  CV = 0.7139 | test set = 0.6916
Specificity        : train set = 0.6304  |  CV = 0.5793 | test set = 0.6012
Accuracy           : train set = 0.7027  |  CV = 0.6529 | test set = 0.6512
MCC                : train set = 0.3953  |  CV = 0.2959 | test set = 0.2933
AUC                : train set = 0.7843  |  CV = 0.7019 | test set = 0.7301
balanced accuracy  : train set = 0.6958  |  CV = 0.6466 | test set = 0.6464


In [49]:
# 2. Calculer le Taux de Faux Positifs (FPR) et le Taux de Vrais Positifs (TPR)
# Ces mesures sont définies dans les équations de performance de l'article [cite: 164-184]
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
roc_auc = auc(fpr, tpr)

# 3. Création du graphique (Style Figure 1 de l'article) 
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'Beta (AUC = {roc_auc:.5f})')

# Ligne diagonale représentant un modèle aléatoire
plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')

# Paramètres d'affichage conformes aux résultats de l'étude [cite: 233, 262]
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC Curve - Beta-strand (SVM)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

TypeError: 'float' object is not callable

In [42]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def balanced_accuracy(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return (sensitivity + specificity) / 2

# --- ÉTAPE 4 : FINALISATION DU MODÈLE BETA (N=10, C=1) ---

# 1. Récupération des 10 meilleures caractéristiques identifiées par ExtraTrees
top_10_features = importances.head(10).index.tolist()
print("\nTop 10 features selected for the SVM model with the hyperparameters from the article:")
for i, feat in enumerate(top_10_features, 1):
    print(f"{i:2d}. {feat}")

# 2. Filtrage des données pour ne garder que ces features
X_train_final = X_train[top_10_features]
X_test_final = X_test[top_10_features]

# 3. Entraînement du modèle SVM final avec tes hyperparamètres optimisés
print("\n Training of the SVM model with the hyperparameters from the article...")
final_svm_beta = SVC(
    C=10, 
    kernel='rbf', 
    gamma= 0.1, 
    probability=True, 
    random_state=42
)
final_svm_beta.fit(X_train_final, y_train)

# 4. Évaluation finale pour confirmation
y_pred_train = final_svm_beta.predict(X_train_final)
y_proba_train = final_svm_beta.predict_proba(X_train_final)[:, 1]


scoring = {'accuracy': 'accuracy','sensitivity': 'recall','specificity': make_scorer(specificity_score),'mcc': make_scorer(matthews_corrcoef),'auc': 'roc_auc', 'balanced_accuracy': make_scorer(balanced_accuracy)}
cv = KFold(n_splits=10, shuffle=True, random_state=42)
cv_results = cross_validate(final_svm_beta, X_train_final, y_train, cv=cv, scoring=scoring, n_jobs=-1)

y_pred_test = final_svm_beta.predict(X_test_final)
y_proba_test = final_svm_beta.predict_proba(X_test_final)[:, 1]

print("\n--- Performance Metrics For the SVM Beta-Strand Model with the hyperparameters from the article ---")
print()
print(f"Number of mutations in the train set : {len(y_train)} with {y_train.sum()} drivers and {len(y_train) - y_train.sum()} passengers")
print(f"Number of mutations in the test set : {len(y_test)} with {y_test.sum()} drivers and {len(y_test) - y_test.sum()} passengers")
print()
#metrics on the train set-----------------------------------------------------
train_auc = roc_auc_score(y_train, y_proba_train)
true_positives = confusion_matrix(y_train, y_pred_train)[1, 1]
true_negatives = confusion_matrix(y_train, y_pred_train)[0, 0]
false_positives = confusion_matrix(y_train, y_pred_train)[0, 1]
false_negatives = confusion_matrix(y_train, y_pred_train)[1, 0]
train_accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
train_sensitivity = true_positives / (true_positives + false_negatives)
train_specificity = true_negatives / (true_negatives + false_positives)
train_balanced_accuracy = (train_sensitivity + train_specificity) / 2
train_MCC = (true_positives * true_negatives - false_positives * false_negatives) / np.sqrt((true_positives + false_positives) * (true_positives + false_negatives) * (true_negatives + false_positives) * (true_negatives + false_negatives))

# metrics on the test set-----------------------------------------------------
true_positives = confusion_matrix(y_test, y_pred_test)[1, 1]
true_negatives = confusion_matrix(y_test, y_pred_test)[0, 0]
false_positives = confusion_matrix(y_test, y_pred_test)[0, 1]
false_negatives = confusion_matrix(y_test, y_pred_test)[1, 0]
test_auc = roc_auc_score(y_test, y_proba_test)
test_accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
test_sensitivity = true_positives / (true_positives + false_negatives)
test_specificity = true_negatives / (true_negatives + false_positives)
test_balanced_accuracy = (test_sensitivity + test_specificity) / 2
test_MCC = (true_positives * true_negatives - false_positives * false_negatives) / np.sqrt((true_positives + false_positives) * (true_positives + false_negatives) * (true_negatives + false_positives) * (true_negatives + false_negatives))

print(f"Sensitivity        : train set = {train_sensitivity:.4f}  |  CV = {np.mean(cv_results['test_sensitivity']):.4f} | test set = {test_sensitivity:.4f}")
print(f"Specificity        : train set = {train_specificity:.4f}  |  CV = {np.mean(cv_results['test_specificity']):.4f} | test set = {test_specificity:.4f}")
print(f"Accuracy           : train set = {train_accuracy:.4f}  |  CV = {np.mean(cv_results['test_accuracy']):.4f} | test set = {test_accuracy:.4f}")
print(f"MCC                : train set = {train_MCC:.4f}  |  CV = {np.mean(cv_results['test_mcc']):.4f} | test set = {test_MCC:.4f}")
print(f"AUC                : train set = {train_auc:.4f}  |  CV = {np.mean(cv_results['test_auc']):.4f} | test set = {test_auc:.4f}")
print(f"balanced accuracy  : train set = {train_balanced_accuracy:.4f}  |  CV = {np.mean(cv_results['test_balanced_accuracy']):.4f} | test set = {test_balanced_accuracy:.4f}")


Top 10 features selected for the SVM model with the hyperparameters from the article:
 1.  rsa
 2. risj880101
 3. dG_numeric_diff
 4. JUKT750101_normalize_diff
 5. MAXF760106_normalize
 6. niek910102
 7. Br_normalize_diff
 8. K0_normalize
 9. dHc_numeric_diff
10. dHc_normalize_diff

 Training of the SVM model with the hyperparameters from the article...

--- Performance Metrics For the SVM Beta-Strand Model with the hyperparameters from the article ---

Number of mutations in the train set : 1544 with 854 drivers and 690 passengers
Number of mutations in the test set : 387 with 214 drivers and 173 passengers

Sensitivity        : train set = 0.8244  |  CV = 0.6659 | test set = 0.6822
Specificity        : train set = 0.7609  |  CV = 0.6025 | test set = 0.6185
Accuracy           : train set = 0.7960  |  CV = 0.6373 | test set = 0.6537
MCC                : train set = 0.5866  |  CV = 0.2680 | test set = 0.3004
AUC                : train set = 0.8778  |  CV = 0.6756 | test set = 0.7130
ba

In [43]:
# 2. Calculer le Taux de Faux Positifs (FPR) et le Taux de Vrais Positifs (TPR)
# Ces mesures sont définies dans les équations de performance de l'article [cite: 164-184]
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
roc_auc = auc(fpr, tpr)

# 3. Création du graphique (Style Figure 1 de l'article) 
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'Beta (AUC = {roc_auc:.5f})')

# Ligne diagonale représentant un modèle aléatoire
plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')

# Paramètres d'affichage conformes aux résultats de l'étude [cite: 233, 262]
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC Curve - Beta-strand (SVM with article hyperparameters)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

TypeError: 'float' object is not callable